# Plantilla de algoritmo. Parte 3 (Problema del Viajante)

**Cómo usarla:** copia este archivo como `parte3/<algoritmo>_<tunombre>.ipynb` (por ejemplo, `parte3/fuerza_bruta_pablo.ipynb`) y trabaja solo **debajo de la línea ✂️**. Lo de arriba (preparación) es el mismo código de la Parte 2 del trabajo y sirve para cargar las versiones del grafo: **no lo modifiques**. La guía completa está en `CONTRIBUTING.md`.

> ⚙️ El notebook se ejecuta desde su carpeta (`parte3/`): la primera celda sube a la raíz del proyecto para que las rutas (`data/...`) funcionen.

In [1]:
# Sube a la raíz del proyecto para que las rutas (data/ciudades.csv, data/versiones/...) funcionen
import os
from pathlib import Path
if Path.cwd().name == "parte3":
    os.chdir("..")

In [2]:
# Librerías que se usan en todo el notebook
import csv                       # leer y escribir tablas en formato CSV
import itertools                 # recorrer todas las combinaciones de parámetros
import unicodedata               # quitar acentos de los nombres (Cataluña -> cataluna)
from pathlib import Path         # rutas de archivos

import networkx as nx            # crear y analizar grafos (la misma librería de las prácticas)
import numpy as np               # tablas de números (las tablas de distancias)
from scipy.sparse import csr_matrix                 # matriz del grafo para el algoritmo de Dijkstra
from scipy.sparse.csgraph import dijkstra           # caminos mínimos desde varios orígenes
from scipy.spatial import cKDTree                   # buscar rápido el nodo más cercano a un punto

from tsp_utils import dibujar_red  # función de dibujo de la red de carreteras, comentada en tsp_utils.py

In [3]:
# ── Parámetros de las versiones (editables) ──
X_KM  = 10             # una ciudad a más de X_KM km de la red se deja fuera de la versión
N_MIN = 4              # mínimo de ciudades para una versión por comunidad autónoma
RECALCULAR = False     # True: vuelve a calcular las 16 tablas (necesita el grafo G de la Parte 1)
CARPETA = Path("data/versiones")     # aquí se guardan las tablas ya calculadas

# Qué tipos de vía (`highway`) puede usar cada versión de carreteras
CARRETERAS = {
    "completo":             ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link", "secondary", "secondary_link"],
    "autovias_principales": ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link"],
    "solo_autovias":        ["motorway", "motorway_link"],
    "sin_autovias":         ["trunk", "trunk_link", "primary", "primary_link",
                             "secondary", "secondary_link"],
}
PESOS = {"km": ("length", 1000), "tiempo": ("travel_time", 1)}   # peso -> (atributo de la arista, divisor)
CONJUNTOS = ["capitales", "capitales_y_100k"]                    # qué ciudades se visitan
GENERALES = ["multigrafo_dirigido", "grafo_dirigido",
             "multigrafo_no_dirigido", "grafo_simple_no_dirigido"]

In [4]:
# Lee la lista de ciudades (nombre, comunidad, si es capital de provincia, coordenadas...)
with open("data/ciudades.csv", encoding="utf-8") as f:
    CIUDADES = list(csv.DictReader(f))
for c in CIUDADES:
    c["lat"], c["lon"] = float(c["lat"]), float(c["lon"])
    c["capital"] = c["capital_provincia"] == "True"

def slug(texto):
    # "Castilla y León" -> "castilla_y_leon" (sin acentos ni espacios, para usarlo como nombre)
    sin_acentos = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return sin_acentos.lower().replace(" ", "_").replace("-", "_")

AMBITOS = ["espana"] + sorted({slug(c["comunidad"]) for c in CIUDADES})

def ciudades_de(nodos, ambito):
    # Nombres de las ciudades de un conjunto: nodos = "capitales" o "capitales_y_100k";
    # ambito = "espana" o una comunidad (p. ej. "galicia")
    return [c["nombre"] for c in CIUDADES
            if (c["capital"] or nodos == "capitales_y_100k")
            and (ambito == "espana" or slug(c["comunidad"]) == ambito)]

print(f"{len(CIUDADES)} ciudades | {len(ciudades_de('capitales', 'espana'))} capitales | ámbitos: {len(AMBITOS) - 1} comunidades")
print("Capitales de Galicia:", ciudades_de("capitales", "galicia"))

75 ciudades | 47 capitales | ámbitos: 15 comunidades
Capitales de Galicia: ['A Coruña', 'Ourense', 'Lugo', 'Pontevedra']


In [5]:
def nombre_tabla(aristas, peso, dirigido):
    return CARPETA / f"{aristas}__{peso}__{'dirigido' if dirigido else 'no_dirigido'}.csv"

def guardar_tabla(D, nombres, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        np.savetxt(f, D, delimiter=",", header=",".join(nombres), comments="", fmt="%.3f")

def leer_tabla(aristas, peso, dirigido):
    ruta = nombre_tabla(aristas, peso, dirigido)
    with open(ruta, encoding="utf-8") as f:
        nombres = f.readline().strip().split(",")        # primera fila: nombres de las ciudades
    return np.loadtxt(ruta, delimiter=",", skiprows=1, encoding="utf-8"), nombres

def escribir_csv(ruta, filas, campos):
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        w.writeheader()
        w.writerows(filas)

In [6]:
UNIDAD = {"km": "km", "tiempo": "segundos"}

def cargar_version(aristas, nodos, ambito, peso, general):
    # aristas: "completo" | "autovias_principales" | "solo_autovias" | "sin_autovias"
    # nodos:   "capitales" | "capitales_y_100k"       ambito: "espana" | "galicia" | "andalucia" ...
    # peso:    "km" | "tiempo"                        general: uno de GENERALES
    # (multigrafo y grafo simple dan el mismo grafo de ciudades: no hay aristas repetidas)
    dirigido = "no_dirigido" not in general
    D, nombres = leer_tabla(aristas, peso, dirigido)
    elegidas = [c for c in ciudades_de(nodos, ambito) if c in nombres]    # sin las que quedaron fuera
    if ambito != "espana" and len(elegidas) < N_MIN:
        raise ValueError(f"Solo hay {len(elegidas)} ciudades (mínimo {N_MIN}) en {ambito}")
    K = nx.DiGraph() if dirigido else nx.Graph()
    for c in CIUDADES:
        if c["nombre"] in elegidas:
            K.add_node(c["nombre"], lat=c["lat"], lon=c["lon"])
    for p, a in enumerate(elegidas):
        for q, b in enumerate(elegidas):
            if p != q and (dirigido or p < q):                            # no dirigido: una arista por par
                K.add_edge(a, b, peso=float(D[nombres.index(a), nombres.index(b)]))
    K.graph.update(aristas=aristas, nodos=nodos, ambito=ambito, peso=peso,
                   unidad=UNIDAD[peso], general=general,
                   fuera=[c for c in ciudades_de(nodos, ambito) if c not in nombres])
    return K

In [7]:
# [AYUDA COMPARTIDA] Funciones comunes a todos los algoritmos
import time
import tracemalloc

def coste_ruta(K, ruta):
    # Coste total del ciclo: suma de las distancias entre ciudades consecutivas y de la última a la primera
    return sum(K[ruta[i]][ruta[(i + 1) % len(ruta)]]["peso"] for i in range(len(ruta)))

def validar_ruta(K, ruta):
    # Comprueba que la ruta visita cada ciudad de K exactamente una vez
    assert len(ruta) == K.number_of_nodes() and set(ruta) == set(K.nodes), "la ruta no visita cada ciudad una vez"

def medir(algoritmo, K, memoria=False, **opciones):
    # Ejecuta algoritmo(K, **opciones), comprueba el resultado y devuelve cuánto ha tardado
    # (con memoria=True mide también el pico de memoria en una segunda ejecución, porque medirla ralentiza)
    t0 = time.perf_counter()
    ruta, coste = algoritmo(K, **opciones)
    segundos = time.perf_counter() - t0
    validar_ruta(K, ruta)
    assert abs(coste - coste_ruta(K, ruta)) < 1e-6 * max(1.0, abs(coste)), "el coste devuelto no coincide con el de la ruta"
    resultado = {"algoritmo": algoritmo.__name__, "ciudades": K.number_of_nodes(), "coste": round(coste, 3),
                 "unidad": K.graph.get("unidad"), "segundos": round(segundos, 4)}
    if memoria:
        tracemalloc.start()
        algoritmo(K, **opciones)
        _, pico = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        resultado["memoria_mb"] = round(pico / 1e6, 2)
    return resultado

---
### ✂️ A PARTIR DE AQUÍ SE INTEGRA EN EL TRABAJO

Todo lo que escribas **debajo de esta línea** pasará al notebook principal (`trabajo_tsp.ipynb`, Parte 3). No modifiques esta celda.

### Held–Karp (programación dinámica, exacto)

<div style="border-left:5px solid #4C72B0;color:#000;background:#EEF2FA;padding:10px 14px;margin:10px 0;border-radius:4px;">

**📚 TEORÍA**

Es el algoritmo **exacto** más conocido que mejora a la fuerza bruta. Esta prueba todos los ciclos posibles, `(n−1)!`, pero **muchos de ellos comparten tramos**: si dos recorridos empiezan en el inicio, pasan por las mismas ciudades (en distinto orden) y terminan en la misma, solo importa cuál de los dos tramos es más corto. Held y Karp (1962) y Bellman (1962) aprovecharon esa idea con **programación dinámica**: se resuelve cada subproblema **una sola vez**, se guarda el resultado y se reutiliza.

**Idea.** Se fija una ciudad de inicio (la 0). Para cada **subconjunto** `S` de las demás ciudades y cada ciudad `j` de ese subconjunto, se define

> `dp[S][j]` = coste del camino más corto que **sale del inicio, visita exactamente las ciudades de `S` y termina en `j`**.

Para llegar a `j` habiendo visitado `S`, el viajante tuvo que estar antes en alguna ciudad `i` de `S∖{j}` habiendo visitado ya `S∖{j}`. Luego

> `dp[S][j] = mínimo sobre i ∈ S∖{j} de ( dp[S∖{j}][i] + coste(i → j) )`

y el recorrido óptimo se obtiene cerrando el ciclo con la vuelta al inicio: `mínimo sobre j de ( dp[todas][j] + coste(j → inicio) )`. Para reconstruir la ruta se retrocede desde el mejor `j`, eligiendo en cada paso el `i` que dio el mínimo.

**Pseudocódigo.**

<pre style="background:#fff;border:1px solid #ccd;padding:10px 14px;border-radius:4px;font-size:13px;">
HELD_KARP(K, inicio):
    ciudades ← [inicio] + las demás ;  m ← n − 1            # las m ciudades que se ordenan
    dp[S][j] ← ∞ para todo subconjunto S y toda ciudad j
    para cada ciudad j:   dp[{j}][j] ← coste(inicio → j)       # subconjuntos de una ciudad

    para k desde 2 hasta m:                                    # tamaño del subconjunto
        para cada subconjunto S de tamaño k, y cada j en S:
            dp[S][j] ← mínimo sobre i en S∖{j} de dp[S∖{j}][i] + coste(i → j)

    j* ← la ciudad j que minimiza dp[todas][j] + coste(j → inicio)
    reconstruir la ruta hacia atrás desde j* ;  devolver (ruta, coste mínimo)
</pre>

**Implementación.** Cada subconjunto se guarda como un número entero (una *máscara de bits*: el bit `j` vale 1 si la ciudad `j` está dentro), así `dp` es una tabla de `2^m` filas y `m` columnas. Para calcular los subconjuntos de tamaño `k` solo hacen falta los de tamaño `k−1`; con `numpy` se procesan **todos a la vez** (sin bucles de Python sobre los `2^m` subconjuntos), que es lo que lo hace usable hasta ~20 ciudades.

**Complejidad.** **Tiempo: O(n²·2ⁿ)** — hay `2^(n−1)·(n−1)` estados y cada uno se calcula probando hasta `n−1` ciudades anteriores. Frente al `O(n!)` de la fuerza bruta es un salto enorme (para `n = 20`: unos `4·10⁸` operaciones frente a `10¹⁷` ciclos), pero sigue siendo exponencial. **Memoria: O(n·2ⁿ)** — es el verdadero límite: la tabla `dp` ocupa `8·(n−1)·2^(n−1)` bytes (≈ 80 MB con 20 ciudades, ≈ 170 MB con 21, ≈ 740 MB con 22). Por eso la función se niega (`ValueError`) a trabajar con más de 21 ciudades.

**Propiedades.**

- **Tipo:** **exacto**. Devuelve siempre un recorrido óptimo.
- **¿Determinista?** **Sí**: no usa azar. Si hay varios óptimos con el mismo coste, devuelve uno de ellos de forma repetible.
- **¿Garantía de calidad?** La mejor posible: el **óptimo**.
- **¿Necesita simetría?** **No.** Usa `coste(i → j)` en el sentido del viaje, así que sirve para el grafo **dirigido** y el **no dirigido**.
- **¿Hasta cuántas ciudades?** **Unas 20–21** en un portátil normal (tarda ~1 s con 20 y la memoria se duplica con cada ciudad nueva). Con 47 capitales no cabe en ningún ordenador: para eso hacen falta otros métodos exactos (como DFJ, programación entera con restricciones de subciclo) o heurísticas.
- **Uso en el trabajo:** da el **óptimo de referencia** para instancias de hasta 20 ciudades, con el que medir cuánto se alejan las heurísticas (vecino más cercano, 2-opt, Christofides, genético...).

</div>

**💻 CÓDIGO**

La función recibe el grafo `K` y devuelve `(ruta, coste)`, como pide el contrato de la Parte 3. Primero ordena las ciudades poniendo el inicio en primer lugar y construye la matriz de costes `W`; después rellena la tabla `dp` por tamaños de subconjunto y, por último, reconstruye la ruta hacia atrás. Si le pasas más de `max_ciudades` ciudades, lanza un error claro antes de intentar reservar memoria.

In [8]:
def held_karp(K, inicio=None, max_ciudades=21):
    # Held-Karp (programación dinámica) para el problema del viajante. Contrato: K es el grafo de
    # ciudades (completo, K[a][b]["peso"] = distancia de a a b) y devuelve (ruta, coste): `ruta` es la
    # lista con todas las ciudades, cada una una vez, en el orden en que se visitan (el viajante vuelve
    # después a la primera) y `coste` es el coste del ciclo, incluida la vuelta. Es EXACTO y no asume
    # simetría: sirve para nx.DiGraph y nx.Graph. Como usa 2^(n-1)·(n-1) números en memoria, se niega
    # a trabajar con más de `max_ciudades` ciudades (21 ≈ 170 MB).

    ciudades = list(K.nodes)
    n = len(ciudades)

    # Casos triviales y comprobaciones
    if n == 0:
        return [], 0.0
    if inicio is None:
        inicio = ciudades[0]                         # por defecto, la primera ciudad del grafo
    elif inicio not in K:
        raise KeyError(f"La ciudad de inicio {inicio!r} no está en el grafo")
    if n > max_ciudades:
        raise ValueError(f"Held-Karp necesita memoria exponencial: {n} ciudades superan el límite de {max_ciudades}")
    orden = [inicio] + [c for c in ciudades if c != inicio]   # la ciudad 0 es siempre el inicio
    if n == 1:
        return orden, 0.0

    # Matriz de costes W[i][j] = coste de ir de la ciudad i a la j (más rápida que consultar el grafo)
    W = np.array([[0.0 if a == b else K[a][b]["peso"] for b in orden] for a in orden])
    m = n - 1                                        # ciudades que hay que ordenar (todas menos el inicio)
    Wi = W[1:, 1:]                                   # costes entre esas m ciudades

    # Un subconjunto de esas m ciudades se guarda como un número entero (máscara de bits):
    # el bit j vale 1 si la ciudad j+1 está en el subconjunto. Hay 2^m subconjuntos.
    N = 1 << m
    mascaras = np.arange(N)

    # dp[S][j] = coste del camino más corto que sale del inicio, visita exactamente las ciudades
    # del subconjunto S y termina en la ciudad j (que debe estar en S). Empieza todo en infinito.
    dp = np.full((N, m), np.inf)
    for j in range(m):
        dp[1 << j, j] = W[0, j + 1]                  # subconjunto de una ciudad: ir directo desde el inicio

    # Agrupamos los subconjuntos por tamaño (número de bits a 1): para calcular los de tamaño k
    # solo hacen falta los de tamaño k-1, así que se pueden procesar todos de golpe con numpy
    cuenta = np.zeros(N, dtype=np.int8)
    for j in range(m):
        cuenta += ((mascaras >> j) & 1).astype(np.int8)
    por_tamano = [mascaras[cuenta == k] for k in range(m + 1)]

    for k in range(2, m + 1):
        for j in range(m):
            S = por_tamano[k]
            S = S[((S >> j) & 1) == 1]               # subconjuntos de tamaño k que contienen la ciudad j
            anterior = S ^ (1 << j)                  # el mismo subconjunto sin j
            # Se llega a j desde alguna ciudad i del subconjunto anterior: se toma la mejor.
            # Las ciudades que no están en `anterior` valen infinito, así que nunca ganan.
            dp[S, j] = (dp[anterior] + Wi[:, j]).min(axis=1)

    # Cerramos el ciclo: el mejor final es el que, sumando la vuelta al inicio, resulta más barato
    completo = N - 1
    cierre = dp[completo] + W[1:, 0]
    j = int(np.argmin(cierre))
    coste = float(cierre[j])

    # Reconstrucción hacia atrás: en cada paso, qué ciudad i precedía a j en el camino óptimo
    indices = []
    S = completo
    while S:
        indices.append(j + 1)
        anterior = S ^ (1 << j)
        if anterior:
            j = int(np.argmin(dp[anterior] + Wi[:, j]))
        S = anterior
    ruta = [orden[0]] + [orden[i] for i in reversed(indices)]
    return ruta, coste

**Prueba rápida.** Tres comprobaciones:

1. **Versión pequeña real:** las 6 ciudades de Galicia empezando en Vigo, con `validar_ruta` y `medir` (tiempo y memoria).
2. **Referencia independiente:** la fuerza bruta (probar los `5! = 120` ciclos que salen de Vigo, que son todos los ciclos posibles) debe dar **exactamente** el mismo coste.
3. **Un límite claro:** con 22 ciudades la función debe negarse en vez de agotar la memoria.

In [9]:
# ── 1. Versión pequeña real: Galicia (6 ciudades), empezando en Vigo ──
K = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="galicia", peso="km", general="grafo_dirigido")
ruta, coste = held_karp(K, inicio="Vigo")
validar_ruta(K, ruta)                               # comprueba que la ruta visita cada ciudad una vez
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print(medir(held_karp, K, memoria=True, inicio="Vigo"))

# ── 2. Referencia independiente: fuerza bruta (todos los ciclos que empiezan en Vigo) ──
resto = [c for c in K.nodes if c != "Vigo"]
optimo = min(coste_ruta(K, ["Vigo"] + list(p)) for p in itertools.permutations(resto))
print(f"\nfuerza bruta: {optimo:.3f} km | Held-Karp: {coste:.3f} km | diferencia: {coste - optimo:+.6f} km")

# ── 3. Más de 21 ciudades: el algoritmo se niega ──
Kgrande = cargar_version(aristas="completo", nodos="capitales", ambito="espana", peso="km", general="grafo_dirigido")
try:
    held_karp(Kgrande.subgraph(list(Kgrande.nodes)[:22]).copy())
except ValueError as error:
    print("\nValueError:", error)

Vigo → Ourense → Lugo → A Coruña → Santiago de Compostela → Pontevedra | coste: 460.4 km
{'algoritmo': 'held_karp', 'ciudades': 6, 'coste': 460.438, 'unidad': 'km', 'segundos': 0.0006, 'memoria_mb': 0.01}

fuerza bruta: 460.438 km | Held-Karp: 460.438 km | diferencia: +0.000000 km

ValueError: Held-Karp necesita memoria exponencial: 22 ciudades superan el límite de 21


**Corrección con más ciudades, en grafo dirigido y no dirigido.** Se toman subgrafos de las `n` primeras ciudades de Andalucía (de 5 a 9) y se comparan, **en las dos versiones**, Held–Karp con la fuerza bruta. En el grafo dirigido la ida y la vuelta pueden costar distinto (sentidos únicos), así que el óptimo de ambas versiones no tiene por qué coincidir.

In [10]:
# ── Held-Karp frente a fuerza bruta en subgrafos de Andalucía ──
print(f"{'general':>26} | {'ciudades':>8} | {'fuerza bruta':>12} | {'Held-Karp':>10} | coincide")
for general in ("grafo_dirigido", "grafo_simple_no_dirigido"):
    KAnd = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="andalucia", peso="km", general=general)
    for n in range(5, 10):
        Kn = KAnd.subgraph(list(KAnd.nodes)[:n]).copy()
        primera = list(Kn.nodes)[0]
        fb = min(coste_ruta(Kn, [primera] + list(p)) for p in itertools.permutations(list(Kn.nodes)[1:]))
        _, hk = held_karp(Kn)
        print(f"{general:>26} | {n:>8} | {fb:>12.3f} | {hk:>10.3f} | {'sí' if abs(fb - hk) < 1e-6 else 'NO'}")

                   general | ciudades | fuerza bruta |  Held-Karp | coincide
            grafo_dirigido |        5 |      857.134 |    857.134 | sí
            grafo_dirigido |        6 |     1038.186 |   1038.186 | sí
            grafo_dirigido |        7 |     1072.204 |   1072.204 | sí
            grafo_dirigido |        8 |     1220.710 |   1220.710 | sí


            grafo_dirigido |        9 |     1226.422 |   1226.422 | sí
  grafo_simple_no_dirigido |        5 |      853.659 |    853.659 | sí
  grafo_simple_no_dirigido |        6 |     1034.622 |   1034.622 | sí
  grafo_simple_no_dirigido |        7 |     1068.258 |   1068.258 | sí
  grafo_simple_no_dirigido |        8 |     1213.427 |   1213.427 | sí


  grafo_simple_no_dirigido |        9 |     1214.792 |   1214.792 | sí


**Escalabilidad.** Se mide Held–Karp sobre las `n` primeras capitales de España (de 8 a 20, versión `completo`/`km`/dirigido). El tiempo y la memoria **se multiplican por ~2 con cada ciudad añadida** (la teoría dice `2ⁿ`); como `medir(..., memoria=True)` ejecuta el algoritmo dos veces, esta celda tarda unos segundos.

In [11]:
# ── Escalabilidad: cuánto tardan y cuánta memoria usan 8, 10, ... 20 ciudades ──
print(f"{'ciudades':>8} | {'coste (km)':>10} | {'segundos':>9} | {'memoria (MB)':>12} | {'tabla dp (MB)':>13}")
for n in (8, 10, 12, 14, 16, 18, 20):
    Kn = Kgrande.subgraph(list(Kgrande.nodes)[:n]).copy()
    r = medir(held_karp, Kn, memoria=True)
    tabla = 8 * (n - 1) * 2 ** (n - 1) / 1e6            # tamaño teórico de la tabla dp
    print(f"{n:>8} | {r['coste']:>10.1f} | {r['segundos']:>9.3f} | {r['memoria_mb']:>12.1f} | {tabla:>13.1f}")

ciudades | coste (km) |  segundos | memoria (MB) | tabla dp (MB)
       8 |     2286.2 |     0.002 |          0.0 |           0.0
      10 |     2661.7 |     0.003 |          0.1 |           0.0
      12 |     2743.6 |     0.007 |          0.3 |           0.2
      14 |     3427.2 |     0.007 |          1.3 |           0.9
      16 |     3486.0 |     0.044 |          5.4 |           3.9


      18 |     3671.7 |     0.233 |         23.8 |          17.8


      20 |     3787.9 |     1.243 |        104.2 |          79.7


<div style="border-left:5px solid #55A868;color:#000;background:#EAF6ED;padding:10px 14px;margin:10px 0;border-radius:4px;">

**🧮 RESULTADOS**

**Prueba rápida (Galicia, 6 ciudades, desde Vigo).** Ruta devuelta:

> **Vigo → Ourense → Lugo → A Coruña → Santiago de Compostela → Pontevedra → Vigo = 460,4 km**

idéntica en coste a la **fuerza bruta** (460,438 km, diferencia 0,000000). `validar_ruta` confirma que visita cada ciudad una vez, y con 22 ciudades la función se niega con un `ValueError` en lugar de agotar la memoria.

**Corrección.** En los 10 subgrafos probados de Andalucía (5 a 9 ciudades, dirigido y no dirigido) Held–Karp da **exactamente el mismo coste que la fuerza bruta**. En el dirigido el óptimo es algo mayor que en el no dirigido (p. ej. con 9 ciudades, 1.226,4 km frente a 1.214,8 km), porque las vías de sentido único hacen que la ida y la vuelta no cuesten lo mismo.

**Escalabilidad** (capitales de España, `completo`/`km`/dirigido):

| Ciudades | Coste (km) | Tiempo | Memoria (pico) | Tabla `dp` teórica |
|---:|---:|---:|---:|---:|
| 8 | 2.286,2 | 0,001 s | 0,0 MB | 0,0 MB |
| 12 | 2.743,6 | 0,002 s | 0,3 MB | 0,2 MB |
| 16 | 3.486,0 | 0,037 s | 5,4 MB | 3,9 MB |
| 18 | 3.671,7 | 0,26 s | 23,8 MB | 17,8 MB |
| 20 | 3.787,9 | 1,13 s | 104,2 MB | 79,7 MB |

Cada ciudad añadida multiplica la memoria por ~2 y el tiempo por 4–5 (la teoría dice `n²·2ⁿ`). **Límite práctico: unas 20–21 ciudades** (≈ 1 s y 100 MB con 20; con 21, estimado, ≈ 5 s y 200 MB; con 22 la tabla ya pasaría de 700 MB). Con las 47 capitales o las 75 ciudades **no es aplicable**: para esas instancias el óptimo hay que buscarlo con DFJ o aproximarlo con heurísticas.

</div>

---